# CrisisLens: The Overlooked Crisis Index
### Full Data Pipeline — Hacklytics 2026
**Databricks x United Nations Geo-Insight Challenge**

This notebook documents the complete data pipeline: downloading UN humanitarian datasets,
cleaning and merging them, computing the Overlooked Crisis Index (OCI), and running
efficiency outlier detection on CBPF projects.

---

## OCI Formula

```
severity_weight = ocha_severity / 5.0
pin_normalized  = people_in_need / total_population
funding_gap     = 1 - (actual_funding / total_requirements)

OCI = pin_normalized × severity_weight × funding_gap
```

Higher OCI = more overlooked. Normalized 0-1 across all crises.

In [ ]:
# Install dependencies (run once)
# %pip install pandas numpy scipy scikit-learn plotly requests

In [ ]:
import pandas as pd
import numpy as np
import requests
import plotly.express as px
from pathlib import Path
from scipy.stats import zscore

print(f"pandas {pd.__version__}, numpy {np.__version__}")

## Step 1: Download Raw Datasets

We pull from four UN data sources:
1. **HNO** (Humanitarian Needs Overview) — People in Need per country/year
2. **FTS** (Financial Tracking Service) — Funding requirements and actuals
3. **COD-PS** (Common Operational Datasets) — Population by country
4. **CBPF** (Country Based Pooled Funds) — Project-level budget and beneficiary data

In [ ]:
# HNO Data — union 2024, 2025, 2026
HNO_URLS = {
    2026: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/edb91329-0e6b-4ebc-b6cb-051b2a11e536/download/hpc_hno_2026.csv",
    2025: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/22093993-e23b-45c8-b84f-61e4a414ebbb/download/hpc_hno_2025.csv",
    2024: "https://data.humdata.org/dataset/8326ed53-8f3a-47f9-a2aa-83ab4ecee476/resource/8e3931a5-452b-4583-9d02-2247a34e397b/download/hpc_hno_2024.csv",
}

hno_frames = []
for year, url in HNO_URLS.items():
    df = pd.read_csv(url, skiprows=[1], low_memory=False)
    df["year"] = year
    hno_frames.append(df)
    print(f"HNO {year}: {len(df)} rows, {len(df.columns)} columns")

df_hno_raw = pd.concat(hno_frames, ignore_index=True)
print(f"\nCombined HNO: {len(df_hno_raw)} rows")
df_hno_raw.columns.tolist()

In [ ]:
# FTS Global Data
FTS_URL = "https://data.humdata.org/dataset/b2bbb33c-2cfb-4809-8dd3-6bbdc080cbb9/resource/b3232da8-f1e4-41ab-9642-b22dae10a1d7/download/fts_requirements_funding_global.csv"
df_fts_raw = pd.read_csv(FTS_URL, skiprows=[1], low_memory=False)
print(f"FTS Global: {len(df_fts_raw)} rows")
df_fts_raw.head()

In [ ]:
# Population Data
POP_URL = "https://data.humdata.org/dataset/27e3d1c6-c57a-4159-85a4-adb6b7aca6b9/resource/d4ea8fba-3d98-4d6e-85c8-84a5b0b4ebd9/download/cod_population_admin0.csv"
df_pop_raw = pd.read_csv(POP_URL, encoding="utf-8-sig", low_memory=False)
print(f"Population: {len(df_pop_raw)} rows")
df_pop_raw.head()

## Step 2: Clean HNO Data

Filter to country-level aggregates (cluster=ALL, no demographic breakdown).
Handle schema differences between 2024 (16 columns) and 2026 (9 columns).

In [ ]:
# Rename and filter HNO
df_hno = df_hno_raw.rename(columns={
    "Country ISO3": "country_iso3",
    "Cluster": "cluster_code",
    "Category": "category",
    "In Need": "people_in_need_raw",
    "Targeted": "people_targeted_raw",
})

# Country-level totals only
mask_all = df_hno["cluster_code"].astype(str).str.strip().str.upper() == "ALL"
mask_no_cat = df_hno["category"].isna() | (df_hno["category"].astype(str).str.strip() == "")

if "Admin 1 PCode" in df_hno.columns:
    mask_country = df_hno["Admin 1 PCode"].isna() | (df_hno["Admin 1 PCode"].astype(str).str.strip() == "")
else:
    mask_country = pd.Series(True, index=df_hno.index)

df_hno_clean = df_hno[mask_all & mask_no_cat & mask_country].copy()

# Convert to numeric
for col in ["people_in_need_raw", "people_targeted_raw"]:
    df_hno_clean[col] = pd.to_numeric(
        df_hno_clean[col].astype(str).str.replace(",", ""), errors="coerce"
    )

df_hno_clean["people_in_need_k"] = df_hno_clean["people_in_need_raw"] / 1000

# Deduplicate
df_hno_clean = df_hno_clean.sort_values("people_in_need_raw", ascending=False)
df_hno_clean = df_hno_clean.drop_duplicates(subset=["country_iso3", "year"])

print(f"HNO clean: {len(df_hno_clean)} country-year rows")
print(f"Countries: {df_hno_clean['country_iso3'].nunique()}")
print(f"Years: {sorted(df_hno_clean['year'].unique())}")
df_hno_clean[["country_iso3", "year", "people_in_need_k"]].head(10)

## Step 3: Clean FTS Funding Data

Filter to HRP-type plans, compute funding gap.

In [ ]:
df_fts = df_fts_raw.rename(columns={
    "countryCode": "country_iso3",
    "year": "year",
    "name": "plan_name",
    "requirements": "requirements_usd",
    "funding": "funding_usd",
})

# Filter named plans
df_fts = df_fts[df_fts["plan_name"].notna() & (df_fts["plan_name"] != "Not specified")].copy()

for col in ["requirements_usd", "funding_usd"]:
    df_fts[col] = pd.to_numeric(df_fts[col], errors="coerce")

df_fts["year"] = pd.to_numeric(df_fts["year"], errors="coerce")
df_fts["requirements_usd_m"] = df_fts["requirements_usd"] / 1e6
df_fts["funding_usd_m"] = df_fts["funding_usd"] / 1e6

df_fts["funding_gap"] = np.where(
    df_fts["requirements_usd"].notna() & (df_fts["requirements_usd"] > 0),
    1 - (df_fts["funding_usd"].fillna(0) / df_fts["requirements_usd"]),
    np.nan,
)
df_fts["funding_gap"] = df_fts["funding_gap"].clip(0, 1)

df_fts = df_fts.sort_values("requirements_usd", ascending=False).drop_duplicates(
    subset=["country_iso3", "year"]
)

print(f"FTS clean: {len(df_fts)} rows")
print(f"Average funding gap: {df_fts['funding_gap'].mean():.1%}")
df_fts[["country_iso3", "year", "plan_name", "requirements_usd_m", "funding_usd_m", "funding_gap"]].head(10)

## Step 4: Clean Population Data

Aggregate to country-level totals from COD-PS admin0 data.

In [ ]:
df_pop_raw.columns = [c.replace("\ufeff", "").strip() for c in df_pop_raw.columns]
df_pop = df_pop_raw.rename(columns={"ISO3": "country_iso3"})

# Filter to total population
df_pop_total = df_pop[df_pop["Population_group"] == "T_TL"].copy()
df_pop_total["Population"] = pd.to_numeric(df_pop_total["Population"], errors="coerce")
df_pop_total["Reference_year"] = pd.to_numeric(df_pop_total["Reference_year"], errors="coerce")

# Most recent year per country
df_pop_total = df_pop_total.sort_values("Reference_year", ascending=False)
df_pop_total = df_pop_total.drop_duplicates(subset=["country_iso3"])
df_pop_total = df_pop_total.rename(columns={"Population": "total_population"})

print(f"Population: {len(df_pop_total)} countries")
df_pop_total[["country_iso3", "total_population"]].head(10)

## Step 5: Compute OCI Scores

Join HNO + FTS + Population, apply the OCI formula, normalize to [0,1].

In [ ]:
# Join datasets
df_oci = df_hno_clean[["country_iso3", "year", "people_in_need_k"]].merge(
    df_fts[["country_iso3", "year", "plan_name", "requirements_usd_m", "funding_usd_m", "funding_gap"]],
    on=["country_iso3", "year"],
    how="left",
).merge(
    df_pop_total[["country_iso3", "total_population"]],
    on="country_iso3",
    how="left",
)

# OCI components
df_oci["severity_weight"] = 1.0  # Default max severity for all HRP crises

df_oci["people_in_need"] = df_oci["people_in_need_k"] * 1000
df_oci["pin_normalized"] = np.where(
    df_oci["total_population"].notna() & (df_oci["total_population"] > 0),
    df_oci["people_in_need"] / df_oci["total_population"],
    np.nan,
)
df_oci["pin_normalized"] = df_oci["pin_normalized"].clip(0, 1)

df_oci["has_funding_data"] = df_oci["funding_gap"].notna()
df_oci["funding_gap"] = df_oci["funding_gap"].fillna(0.5)

# OCI raw
df_oci["oci_raw"] = df_oci["pin_normalized"] * df_oci["severity_weight"] * df_oci["funding_gap"]
df_oci["oci_raw"] = df_oci["oci_raw"].fillna(0)

# Normalize
oci_min, oci_max = df_oci["oci_raw"].min(), df_oci["oci_raw"].max()
if oci_max > oci_min:
    df_oci["oci_score"] = ((df_oci["oci_raw"] - oci_min) / (oci_max - oci_min)).round(4)
else:
    df_oci["oci_score"] = 0.0

df_oci["oci_rank"] = df_oci["oci_score"].rank(ascending=False, method="min").astype(int)

print(f"OCI computed for {len(df_oci)} country-year rows")
print(f"\nTop 10 Most Overlooked Crises:")
df_oci.nlargest(10, "oci_score")[
    ["country_iso3", "year", "oci_score", "oci_rank", "pin_normalized", "funding_gap", "people_in_need_k", "requirements_usd_m"]
]

## Step 6: OCI Choropleth Visualization

In [ ]:
# Latest year per country
df_latest = df_oci.sort_values("year", ascending=False).drop_duplicates("country_iso3")

fig = px.choropleth(
    df_latest,
    locations="country_iso3",
    locationmode="ISO-3",
    color="oci_score",
    color_continuous_scale=[[0, "#2ecc71"], [0.35, "#f39c12"], [0.65, "#e74c3c"], [1, "#8e0000"]],
    range_color=(0, 1),
    hover_data={"oci_score": ":.3f", "funding_gap": ":.1%", "people_in_need_k": ":,.0f"},
    labels={"oci_score": "OCI Score"},
    title="Overlooked Crisis Index — World Map",
)
fig.update_layout(
    height=500,
    geo=dict(showframe=False, projection_type="natural earth"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## Step 7: Cluster-Level Funding Analysis

In [ ]:
# Load cluster-level FTS data
FTS_CLUSTER_URL = "https://data.humdata.org/dataset/b2bbb33c-2cfb-4809-8dd3-6bbdc080cbb9/resource/80975d5b-508b-47b2-a10c-b967104d3179/download/fts_requirements_funding_cluster_global.csv"
df_cluster = pd.read_csv(FTS_CLUSTER_URL, skiprows=[1], low_memory=False)
df_cluster = df_cluster.rename(columns={
    "countryCode": "country_iso3",
    "cluster": "cluster_name",
    "requirements": "requirements_usd",
    "funding": "funding_usd",
})

df_cluster = df_cluster[df_cluster["cluster_name"].notna()].copy()
for col in ["requirements_usd", "funding_usd"]:
    df_cluster[col] = pd.to_numeric(df_cluster[col], errors="coerce")

df_cluster["funding_gap"] = np.where(
    df_cluster["requirements_usd"] > 0,
    1 - df_cluster["funding_usd"].fillna(0) / df_cluster["requirements_usd"],
    np.nan,
).clip(0, 1)

# Most underfunded clusters globally
cluster_avg = df_cluster.groupby("cluster_name")["funding_gap"].mean().sort_values(ascending=False)
print("Average funding gap by cluster (globally):")
print(cluster_avg.head(15).to_string())

## Step 8: CBPF Project Efficiency Analysis

In [ ]:
# Load CBPF project data
import json

CBPF_URL = "https://cbpfapi.unocha.org/vo1/odata/ProjectSummary?ShowAllPooledFunds=1&$format=json"
resp = requests.get(CBPF_URL, timeout=180)
cbpf_data = resp.json()
records = cbpf_data.get("value", cbpf_data)
df_cbpf = pd.DataFrame(records)

print(f"CBPF projects loaded: {len(df_cbpf)}")

# Clean
df_cbpf["country_iso3"] = df_cbpf["ChfProjectCode"].str[:3].str.upper()
for col in ["Men", "Women", "Boys", "Girls"]:
    df_cbpf[col] = pd.to_numeric(df_cbpf[col], errors="coerce").fillna(0)
df_cbpf["beneficiaries_total"] = df_cbpf["Men"] + df_cbpf["Women"] + df_cbpf["Boys"] + df_cbpf["Girls"]
df_cbpf["Budget"] = pd.to_numeric(df_cbpf["Budget"], errors="coerce")

# Filter to 2020+
df_cbpf["AllocationYear"] = pd.to_numeric(df_cbpf["AllocationYear"], errors="coerce")
df_cbpf = df_cbpf[(df_cbpf["AllocationYear"] >= 2020) & (df_cbpf["Budget"] > 0)].copy()

# Efficiency ratio
df_cbpf["efficiency_ratio"] = np.where(
    df_cbpf["beneficiaries_total"] > 0,
    df_cbpf["beneficiaries_total"] / df_cbpf["Budget"],
    np.nan,
)

print(f"Projects after filtering: {len(df_cbpf)}")
print(f"Efficiency ratio stats:")
df_cbpf["efficiency_ratio"].describe()

## Step 9: Efficiency Outlier Detection

Z-score analysis within clusters. Threshold: |z| > 2.0.

In [ ]:
# Simple keyword-based cluster inference
CLUSTER_KEYWORDS = {
    "Education": "Education", "Health": "Health", "Nutrition": "Nutrition",
    "Protection": "Protection", "Shelter": "Shelter", "WASH": "WASH",
    "Water": "WASH", "Food": "Food Security", "Agriculture": "Food Security",
}

def infer_cluster(title):
    if pd.isna(title): return "Other"
    for kw, cl in CLUSTER_KEYWORDS.items():
        if kw.lower() in str(title).lower(): return cl
    return "Other"

df_cbpf["cluster_name"] = df_cbpf["ProjectTitle"].apply(infer_cluster)

# Z-score within each cluster
df_cbpf["zscore"] = np.nan
df_cbpf["flag"] = "normal"

for cluster, group in df_cbpf.groupby("cluster_name"):
    valid = group["efficiency_ratio"].notna()
    if valid.sum() < 3:
        df_cbpf.loc[group.index, "flag"] = "insufficient_data"
        continue
    
    log_ratios = np.log1p(group.loc[valid.values, "efficiency_ratio"].values)
    z = (log_ratios - log_ratios.mean()) / log_ratios.std(ddof=1)
    
    valid_idx = group.index[valid.values]
    df_cbpf.loc[valid_idx, "zscore"] = z
    df_cbpf.loc[valid_idx[z > 2.0], "flag"] = "high_efficiency"
    df_cbpf.loc[valid_idx[z < -2.0], "flag"] = "low_efficiency"

print("Outlier distribution:")
print(df_cbpf["flag"].value_counts())
print(f"\nBenchmark projects (high efficiency): {(df_cbpf['flag'] == 'high_efficiency').sum()}")

## Step 10: Recommender Demo

Cosine similarity to find comparable benchmark projects.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

# Build feature matrix
df_feat = df_cbpf.dropna(subset=["cluster_name"]).copy()
df_feat["OrganizationType"] = df_feat["OrganizationType"].fillna("Unknown")

enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat = enc.fit_transform(df_feat[["cluster_name", "OrganizationType"]])

df_feat["log_budget"] = np.log1p(df_feat["Budget"].fillna(0))
df_feat["log_ben"] = np.log1p(df_feat["beneficiaries_total"].fillna(0))

scaler = MinMaxScaler()
X_num = sp.csr_matrix(scaler.fit_transform(df_feat[["log_budget", "log_ben"]]))
X = sp.hstack([X_cat, X_num])

print(f"Feature matrix: {X.shape[0]} projects x {X.shape[1]} features")

# Demo: find similar projects to the first project
query_idx = 0
sims = cosine_similarity(X[query_idx], X)[0]
top5 = sims.argsort()[-6:-1][::-1]  # top 5 excluding self

print(f"\nQuery project: {df_feat.iloc[query_idx]['ChfProjectCode']}")
print(f"Top 5 similar projects:")
for idx in top5:
    print(f"  {df_feat.iloc[idx]['ChfProjectCode']}: similarity={sims[idx]:.3f}, "
          f"budget=${df_feat.iloc[idx]['Budget']:,.0f}, "
          f"beneficiaries={df_feat.iloc[idx]['beneficiaries_total']:,.0f}")

## Conclusions and Limitations

### Key Findings
- The OCI reveals significant mismatches between humanitarian needs and funding coverage
- Cluster-level analysis shows that country-level aggregate funding can mask critical underfunding in specific sectors
- Efficiency outlier detection identifies benchmark projects that achieve strong beneficiary coverage relative to budget

### Limitations
- **Severity data:** OCHA severity classifications (1-5) are not in the HNO CSV — we default to maximum severity for all HRP crises
- **Bilateral funding:** The OCI only captures funding tracked through FTS/CBPF — bilateral aid that bypasses these systems is not included
- **Media attention:** The optional GDELT/Google Trends layer is not implemented in this pipeline
- **Cluster inference:** CBPF project cluster assignment uses keyword matching on project titles, which may misclassify some projects
- **Population data:** Reference years vary by country — some population figures may be 2-3 years old

### Data Quality Notes
- HNO schema changes between years (2024/2025 have admin-level detail, 2026 does not)
- Some HRP plans span multiple countries — we filter to single-country plans for per-country analysis
- CBPF beneficiary counts are zero for many pre-2020 projects (data collection improvement over time)